<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-10-production-deployment/lesson-10.7-scaling-ai-apps/notebooks/exercises-10.7.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 10.7 — Scaling AI Apps
Netsetos GenAI Course v2.0 | Module 10: Production Deployment

HPA vs KEDA, continuous batching, tensor parallel, load balancing, spot workers, queue-based shedding.


In [ ]:
print('Module 10: Production Deployment')
print('Lesson 10.7: Scaling AI Apps')


## Ex 1: Autoscaling Triggers


In [ ]:
print('Autoscaling triggers and their fit for LLM workloads:')
for trigger, when_use, limit in [
    ('CPU %','Stateless gateways','Hides per-request work'),
    ('QPS','Uniform short responses','Misleading for long generations'),
    ('Queue depth (KEDA)','Worker pools, async','Lag during burst absorption'),
    ('Tokens-in-flight','LLM serving','Needs custom metric exporter'),
    ('GPU utilization','Inference workers','Set target 70-80% for headroom'),
]:
    print(f'  {trigger:22s}: {when_use:28s} | Limit: {limit}')


## Ex 2: HPA vs KEDA


In [ ]:
print('HPA vs KEDA:')
for feature, hpa, keda in [
    ('Built-in metrics','CPU, memory','same + 50+ event sources'),
    ('Redis queue depth','via adapter','first-class trigger'),
    ('Prometheus query','via adapter','native'),
    ('Scale-to-zero','no (min=1)','yes'),
    ('Kafka lag','via adapter','native'),
    ('Cooldown control','basic','granular per trigger'),
]:
    print(f'  {feature:20s}: HPA={hpa:16s} | KEDA={keda}')


## Ex 3: vLLM Continuous Batching Gain


In [ ]:
print('vLLM continuous batching throughput (Llama-3.1-8B on H100, illustrative):')
for concurrency, naive_toks, vllm_toks in [
    ( 1,  52, 180),
    ( 4,  55, 410),
    ( 8,  60, 640),
    (16,  60, 780),
    (32,  60, 824),
]:
    gain = vllm_toks / naive_toks
    print(f'  concurrency={concurrency:>3} | naive={naive_toks:>3} tok/s | vllm={vllm_toks:>4} tok/s | gain {gain:>5.1f}x')


## Ex 4: Inference Server Comparison


In [ ]:
print('Self-hosted inference servers:')
for server, strength, best_for in [
    ('vLLM','PagedAttention, continuous batching','OSS LLM serving default'),
    ('TGI (HuggingFace)','Easy multi-GPU, HF ecosystem','HF model deployments'),
    ('TensorRT-LLM','Best NVIDIA throughput','Latency-critical NVIDIA stacks'),
    ('Triton','Multi-framework, ensemble graphs','Mixed model types in one server'),
    ('Ray Serve','Python-native, actor model','Multi-model + Python glue'),
]:
    print(f'  {server:18s}: {strength:36s} | Best: {best_for}')


## Ex 5: Load Balancing for LLM


In [ ]:
print('Load balancing strategies:')
for strat, fits_llm, why in [
    ('Round-robin','No','Variable response length skews load'),
    ('Least-connections','OK','Better for long requests'),
    ('Least-loaded (queue-aware)','YES','Best for LLM serving'),
    ('Sticky session','NEVER','Defeats continuous batching'),
    ('Consistent hash on user','Sometimes','Cache locality + session state'),
]:
    print(f'  {strat:28s}: fits_llm={fits_llm:8s} | {why}')


## Ex 6: Multi-GPU Strategies


In [ ]:
print('Multi-GPU strategies:')
for strat, what, when in [
    ('Tensor parallel','Split matmul across GPUs in a node','Model does not fit on one GPU'),
    ('Pipeline parallel','Split layers across nodes','Beyond a single-node limit'),
    ('Sequence parallel','Split long context across GPUs','Very long context'),
    ('Expert parallel','Split MoE experts','MoE models (Mixtral, DeepSeek)'),
    ('Data parallel','Replicate model across GPUs','Throughput scaling'),
]:
    print(f'  {strat:18s}: {what:38s} | When: {when}')


## Ex 7: Capacity Planning Math


In [ ]:
peak_qps = 800
avg_out_tokens = 450
per_gpu_tok_s = 800  # vLLM on H100
headroom = 0.30

peak_tok_s = peak_qps * avg_out_tokens
raw_gpus = peak_tok_s / per_gpu_tok_s
with_headroom = raw_gpus * (1 + headroom)

print('Capacity plan for peak workload:')
print(f'  peak QPS: {peak_qps}')
print(f'  avg output tokens: {avg_out_tokens}')
print(f'  per-GPU throughput: {per_gpu_tok_s} tok/s')
print(f'  peak token rate: {peak_tok_s:,} tok/s')
print(f'  raw GPUs: {raw_gpus:.1f}')
print(f'  with {int(headroom*100)}% headroom: {with_headroom:.1f} GPUs')
print(f'  provision: {int(with_headroom) + 1} GPUs, plus spot burst pool')


---
## Recap
vLLM continuous batching + KEDA on queue depth + queue-based shedding + spot workers + capacity from benchmarks.
Token-rate > QPS for LLM autoscaling. Least-loaded LB. Tensor + pipeline parallel for 70B+.
Spot 60-80% cheaper but needs graceful shutdown + cross-zone spread.
Plan from measured per-GPU tok/s, add 30% headroom, separate burst pool for spikes.
